# Độ phủ tọa độ và so sánh ô H3 tại Jilin

Phân tích dùng tọa độ điểm giao `lng/lat` của các đơn trong LaDe-D Jilin. [H3](https://h3geo.org/docs/) gán mỗi điểm vào một ô địa lý. Hai độ phân giải được so sánh trên cùng tập đơn của toàn kỳ.

## 1. Chất lượng và phân bố tọa độ

Kiểm tra tọa độ thiếu hoặc không phải số, ngoài miền hợp lệ và cặp `(0, 0)`. Lớp bản đồ dùng các đơn có tọa độ qua kiểm tra. Biểu đồ điểm giúp nhận diện các điểm xa cụm chính để xem lại, vì khoảng cách lớn tự nó chưa đủ để kết luận tọa độ sai.

In [ ]:
from pathlib import Path
import sys

import h3
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import LogNorm
from matplotlib.ticker import FormatStrFormatter
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.data_loader import loadRawDelivery
from src.preprocessing import validateDemandOrders
from src.spatial_aggregation import (
    RESOLUTIONS, auditCoordinates, compareResolutions, countOrdersByCell,
)

orders = validateDemandOrders(loadRawDelivery())
mappable, coverage = auditCoordinates(orders)
display(coverage.round(2))
display(mappable[["lng", "lat"]].agg(["min", "max"]).round(5))

fig, ax = plt.subplots(figsize=(7, 6), layout="constrained")
for region, group in mappable.groupby("region_id"):
    ax.scatter(group["lng"], group["lat"], s=3, alpha=0.35, label=str(region))
ax.set(xlabel="Kinh độ (°E)", ylabel="Vĩ độ (°N)", title="Phân bố điểm giao theo vùng")
ax.legend(title="Mã vùng", markerscale=3)
plt.show()

## 2. So sánh H3 mức 8 và 9

Ô ít đơn là ô có từ 1 đến 5 đơn trong toàn kỳ. Tỷ lệ ô ít đơn dùng số ô có dữ liệu làm mẫu số. Tỷ lệ đơn trong các ô ít đơn cho biết các ô này chiếm bao nhiêu trong tổng số đơn được gán. Hai hình dùng cùng phạm vi tọa độ và cùng thang màu log cho số đơn mỗi ô.

In [ ]:
cell_counts = {level: countOrdersByCell(mappable, level) for level in RESOLUTIONS}
comparison = compareResolutions(cell_counts)
if not comparison["mapped_orders"].eq(len(mappable)).all():
    raise ValueError("Số đơn gán vào H3 không khớp với kiểm tra tọa độ.")
display(comparison.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 7), sharex=True, sharey=True, layout="constrained")
norm = LogNorm(vmin=1, vmax=max(2, max(counts.max() for counts in cell_counts.values())))
for ax, (level, counts) in zip(axes, cell_counts.items()):
    polygons = [[(lng, lat) for lat, lng in h3.cell_to_boundary(cell)] for cell in counts.index]
    collection = PolyCollection(polygons, array=counts.to_numpy(), cmap="viridis", norm=norm)
    ax.add_collection(collection, autolim=False)
    ax.set(
        title=f"H3 mức {level} · {len(counts)} ô có đơn",
        xlabel="Kinh độ (°E)",
        xlim=(mappable["lng"].min() - 0.02, mappable["lng"].max() + 0.02),
        ylim=(mappable["lat"].min() - 0.02, mappable["lat"].max() + 0.02),
    )
    ax.set_aspect("equal")
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
axes[0].set_ylabel("Vĩ độ (°N)")
fig.colorbar(collection, ax=axes, label="Số đơn trong ô")
plt.show()

## 3. Kết quả trên LaDe-D Jilin

Cả 31.415 đơn qua kiểm tra tọa độ và được gán vào H3 ở cả hai mức. Biểu đồ cho thấy các cụm giao chính và một số điểm rời xa cụm. Các điểm rời có tọa độ hợp lệ theo tiêu chí trên và được giữ trong phân tích.

Mức 8 cho thấy các cụm chính ở góc nhìn toàn bộ vùng quan sát. Mức 9 tách các cụm đó thành nhiều ô nhỏ hơn. Tỷ lệ đơn trong ô có 1–5 đơn tăng từ 0,69% lên 2,80%. Đây là thống kê toàn kỳ, nên khi lọc khoảng ngày ngắn, số đơn mỗi ô có thể thấp hơn.